# Iris EDA

**Purpose**: Exploratory data analysis for the Iris classification task.  
**Author**: {your name}  
**Date**: {YYYY-MM-DD}  
**Dataset**: Iris (sklearn built-in) — 150 samples, 4 features, 3 classes

---

## Goals
1. Understand the feature distributions and class balance
2. Identify which features most separate the classes
3. Form hypotheses to test in the experiment notebook

> **Rule**: No model training in this notebook. EDA only.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris

SEED = 42
np.random.seed(SEED)

sns.set_theme(style='whitegrid')
%matplotlib inline

## Load Data

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame
df['target_name'] = df['target'].map(dict(enumerate(iris.target_names)))

print(f"Shape: {df.shape}")
df.head()

## Basic Statistics

In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Class Balance ===")
print(df['target_name'].value_counts())

print("\n=== Descriptive Statistics ===")
df.describe()

## Feature Distributions

In [ ]:
feature_cols = iris.feature_names

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    for species in df['target_name'].unique():
        subset = df[df['target_name'] == species]
        axes[i].hist(subset[col], alpha=0.6, label=species, bins=15)
    axes[i].set_title(col)
    axes[i].legend()

plt.suptitle('Feature Distributions by Species', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Correlation Analysis

In [ ]:
corr = df[feature_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('outputs/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nHigh correlations (|r| > 0.7):")
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.7:
            print(f"  {corr.columns[i]} <-> {corr.columns[j]}: {corr.iloc[i, j]:.3f}")

## Pairplot

In [ ]:
g = sns.pairplot(df, hue='target_name', diag_kind='kde', plot_kws={'alpha': 0.6})
g.fig.suptitle('Pairplot — Iris Features', y=1.02)
g.savefig('outputs/pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

## Hypotheses

Based on the EDA above, the following hypotheses will be tested in `02_experiment.ipynb`:

**H1**: Petal length and petal width are the most discriminative features.  
*Reasoning*: The pairplot shows near-perfect separation between setosa and the other two classes on petal features. Versicolor and virginica also show good separation on petal dimensions.

**H2**: A Random Forest classifier will achieve >95% test accuracy using all 4 features.  
*Reasoning*: The three classes show clear separation in the feature space; ensemble methods should handle the versicolor/virginica overlap robustly.

**H3**: Removing sepal features (sepal length, sepal width) will cost less than 2% accuracy.  
*Reasoning*: The correlation heatmap shows sepal width has low correlation with other features and the pairplot shows poor class separation on sepal dimensions.

## EDA Summary

- **No missing values**: clean dataset, no imputation needed
- **Perfectly balanced**: 50 samples per class
- **High petal correlation**: petal_length and petal_width (r=0.96) — may need to monitor for multicollinearity
- **Setosa is linearly separable** from the other two classes on all features
- **Versicolor/virginica overlap** primarily on sepal width — this will be the challenge

→ Proceed to `02_experiment.ipynb` to test H1, H2, H3.